# Intelligent Resume Parser & Job Matching System
**Goal:** Parse a PDF resume, extract useful candidate information, compare it with a job description, and identify matched/missing skills.

**Approach:** PDF text extraction → rule-based entity/skill extraction → TF-IDF similarity + skill coverage → explainable results.

> This is a baseline project, not an automated hiring decision system. Scores are indicative and should not be used as the sole basis for hiring.

## 1. Install dependencies
Run this cell once. If Jupyter asks for a restart after installation, restart the kernel and continue.

In [ ]:
%pip install -q pymupdf pandas numpy scikit-learn matplotlib seaborn

## 2. Imports

In [ ]:
import re
import json
import textwrap
from pathlib import Path

import fitz  # PyMuPDF
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 120)
RANDOM_STATE = 42

## 3. Upload a resume PDF
Choose a resume PDF from your computer. Scanned/image-only PDFs may need OCR; this baseline extracts selectable text.

In [ ]:
from pathlib import Path

RESUME_PATH = input("Enter the full path to your resume PDF: ").strip().strip('"')
resume_path = Path(RESUME_PATH).expanduser()

if not resume_path.exists():
    raise FileNotFoundError(f"File not found: {resume_path}")
if resume_path.suffix.lower() != ".pdf":
    raise ValueError("Please provide a PDF resume.")

print("Resume:", resume_path.name)

## 4. Extract text from PDF

In [ ]:
def extract_pdf_text(pdf_path):
    text_parts = []
    with fitz.open(pdf_path) as doc:
        for page_num, page in enumerate(doc, start=1):
            page_text = page.get_text("text")
            if page_text.strip():
                text_parts.append(f"\n--- PAGE {page_num} ---\n{page_text}")
    return "\n".join(text_parts).strip()

resume_text = extract_pdf_text(resume_path)
print("Characters extracted:", len(resume_text))
print("\nPreview:\n")
print(resume_text[:4000] if resume_text else "No text extracted. This may be a scanned PDF.")

## 5. Provide a job description
Paste a job description below. You can replace the sample with a real job description for testing.

In [ ]:
job_description = '''
Machine Learning Engineer

Responsibilities:
- Build and evaluate machine learning models.
- Develop data pipelines and deploy ML services.
- Work with Python, SQL, pandas, scikit-learn, and Git.
- Communicate results and collaborate with engineering teams.

Requirements:
- Strong Python programming.
- Experience with machine learning and data preprocessing.
- Familiarity with NLP, model evaluation, and REST APIs.
- Bachelor's degree in Computer Science or a related field.
'''.strip()

print(job_description)

## 6. Clean text

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9+#.\s@:/_-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

resume_clean = clean_text(resume_text)
job_clean = clean_text(job_description)

print("Resume cleaned characters:", len(resume_clean))
print("Job description cleaned characters:", len(job_clean))

## 7. Extract common resume fields
These use transparent patterns and may need adjustment for different resume formats. Do not assume every field will be found.

In [ ]:
def first_match(patterns, text, flags=re.IGNORECASE):
    for pattern in patterns:
        match = re.search(pattern, text, flags)
        if match:
            return match.group(1).strip(" \t\n,;|")
    return None

def extract_email(text):
    match = re.search(r"[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}", text, re.I)
    return match.group(0) if match else None

def extract_phone(text):
    candidates = re.findall(r"(?:\+?\d[\d ()-]{8,}\d)", text)
    for candidate in candidates:
        digits = re.sub(r"\D", "", candidate)
        if 10 <= len(digits) <= 15:
            return candidate.strip()
    return None

def extract_links(text):
    urls = re.findall(r"(?:https?://)?(?:www\.)?(?:linkedin\.com|github\.com)/[^\s|,;]+", text, re.I)
    return list(dict.fromkeys(url.rstrip(").,") for url in urls))

candidate_info = {
    "email": extract_email(resume_text),
    "phone": extract_phone(resume_text),
    "links": extract_links(resume_text),
}
pd.DataFrame([candidate_info])

## 8. Skills dictionary
Start with a curated vocabulary. Expand it for the job families you care about. Phrase matching avoids treating every word as a skill.

In [ ]:
SKILLS = [
    "python", "java", "c++", "c", "sql", "javascript", "typescript",
    "machine learning", "deep learning", "natural language processing", "nlp",
    "generative ai", "large language models", "llm", "prompt engineering",
    "computer vision", "pytorch", "tensorflow", "scikit-learn", "pandas",
    "numpy", "matplotlib", "seaborn", "data analysis", "data visualization",
    "statistics", "feature engineering", "model evaluation", "classification",
    "regression", "clustering", "recommendation systems", "transformers",
    "hugging face", "langchain", "rag", "faiss", "vector databases",
    "fastapi", "flask", "streamlit", "rest api", "api development",
    "html", "css", "react", "node.js", "express", "git", "github",
    "postgresql", "mysql", "mongodb", "docker", "aws", "azure", "gcp",
    "data structures", "algorithms", "problem solving", "excel", "power bi",
    "tableau", "etl", "data preprocessing", "communication", "teamwork"
]

def find_skills(text, skill_list=SKILLS):
    normalized = clean_text(text)
    found = []
    for skill in skill_list:
        skill_norm = clean_text(skill)
        pattern = r"(?<![a-z0-9])" + re.escape(skill_norm) + r"(?![a-z0-9])"
        if re.search(pattern, normalized):
            found.append(skill)
    return sorted(set(found), key=str.lower)

resume_skills = find_skills(resume_text)
job_skills = find_skills(job_description)

print("Resume skills:", resume_skills)
print("\nSkills found in job description:", job_skills)

## 9. Match and missing skills

## 10. Prepare semantic matching

The baseline TF-IDF score compares overlapping words. The next step uses a **Sentence Transformer** to encode the resume and job description into dense vectors. This can recognize related wording even when exact terms differ.

The first run downloads the model and may take a few minutes. It requires an internet connection for the initial model download.

In [ ]:
%pip install -q sentence-transformers

## 11. Improve skill matching with aliases

In [ ]:
# Canonicalize common variants so equivalent terms are less likely to be missed.
SKILL_ALIASES = {
    "ml": "machine learning",
    "ai": "artificial intelligence",
    "gen ai": "generative ai",
    "llms": "large language models",
    "large language model": "large language models",
    "natural language processing": "nlp",
    "scikit learn": "scikit-learn",
    "sklearn": "scikit-learn",
    "pytorch": "pytorch",
    "powerbi": "power bi",
    "nodejs": "node.js",
    "restful api": "rest api",
    "apis": "rest api",
    "postgres": "postgresql",
}

def canonicalize_skill(skill):
    key = clean_text(skill)
    return SKILL_ALIASES.get(key, key)

def canonical_skill_set(skills):
    return sorted({canonicalize_skill(s) for s in skills}, key=str.lower)

resume_skills_canonical = canonical_skill_set(resume_skills)
job_skills_canonical = canonical_skill_set(job_skills)

matched_skills = sorted(set(resume_skills_canonical) & set(job_skills_canonical), key=str.lower)
missing_skills = sorted(set(job_skills_canonical) - set(resume_skills_canonical), key=str.lower)
extra_skills = sorted(set(resume_skills_canonical) - set(job_skills_canonical), key=str.lower)

skill_coverage = (
    len(matched_skills) / len(set(job_skills_canonical)) * 100
    if job_skills_canonical else 0.0
)

print(f"Canonical skill coverage: {skill_coverage:.1f}%")
print("Matched skills:", matched_skills)
print("Potential skill gaps:", missing_skills)
print("Additional resume skills:", extra_skills)

## 12. TF-IDF baseline similarity

In [ ]:
# Keep TF-IDF as a useful lexical baseline for comparison.
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=20000,
    sublinear_tf=True
)
tfidf_matrix = tfidf_vectorizer.fit_transform([resume_clean, job_clean])
tfidf_similarity = float(cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0, 0])
tfidf_similarity_pct = 100 * tfidf_similarity

print(f"TF-IDF lexical similarity: {tfidf_similarity_pct:.2f}%")

## 13. Semantic similarity with Sentence Transformers

Model: `sentence-transformers/all-MiniLM-L6-v2`.

This model creates sentence/document embeddings. We normalize embeddings and compute cosine similarity. Similarity is a relevance signal, **not** a probability of selection.

In [ ]:
from sentence_transformers import SentenceTransformer

SEMANTIC_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
semantic_model = SentenceTransformer(SEMANTIC_MODEL_NAME)

# Long resumes/job descriptions can exceed useful input lengths. Encode paragraphs/chunks,
# then compare the average embedding of each document.
def split_into_chunks(text, max_words=180):
    words = text.split()
    if not words:
        return [""]
    return [" ".join(words[i:i + max_words]) for i in range(0, len(words), max_words)]

def document_embedding(text, model=semantic_model):
    chunks = split_into_chunks(text)
    embeddings = model.encode(
        chunks,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )
    vector = np.mean(embeddings, axis=0)
    norm = np.linalg.norm(vector)
    return vector / norm if norm else vector

resume_embedding = document_embedding(resume_text)
job_embedding = document_embedding(job_description)

semantic_similarity = float(np.dot(resume_embedding, job_embedding))
# Use non-negative cosine similarity on a 0–100 display scale.
semantic_similarity_pct = float(np.clip(semantic_similarity, 0, 1) * 100)

print(f"Raw semantic cosine similarity: {semantic_similarity:.4f}")
print(f"Mapped semantic similarity: {semantic_similarity_pct:.2f}%")

## 14. Compare resume sections with the job description

This is a simple section-aware signal. It looks for common section headings, then measures semantic similarity for each section found. Heading detection is heuristic and depends on resume formatting.

In [ ]:
SECTION_PATTERNS = {
    "summary": r"(?im)^\s*(professional summary|summary|profile|objective)\s*:?\s*$",
    "skills": r"(?im)^\s*(technical skills|skills|core competencies|technologies)\s*:?\s*$",
    "experience": r"(?im)^\s*(work experience|professional experience|experience|internships?)\s*:?\s*$",
    "projects": r"(?im)^\s*(projects|academic projects|personal projects)\s*:?\s*$",
    "education": r"(?im)^\s*(education|academic background|qualifications)\s*:?\s*$",
}

def extract_resume_sections(text):
    matches = []
    for section, pattern in SECTION_PATTERNS.items():
        for match in re.finditer(pattern, text):
            matches.append((match.start(), section, match.end()))
    matches.sort()
    sections = {}
    for i, (start, name, content_start) in enumerate(matches):
        end = matches[i + 1][0] if i + 1 < len(matches) else len(text)
        content = text[content_start:end].strip()
        if content:
            sections.setdefault(name, "")
            sections[name] += "\n" + content
    return sections

resume_sections = extract_resume_sections(resume_text)
section_scores = {}

for section_name, section_text in resume_sections.items():
    section_vec = document_embedding(section_text)
    section_scores[section_name] = round(
        float(np.dot(section_vec, job_embedding)), 4
    )

if section_scores:
    section_scores_df = pd.DataFrame(
        [{"Section": k.title(), "Semantic cosine similarity": v}
         for k, v in section_scores.items()]
    ).sort_values("Semantic cosine similarity", ascending=False)
    display(section_scores_df)
else:
    print("No standard section headings were detected. Overall semantic matching still works.")

## 15. Combine signals into an explainable baseline score

In [ ]:
# Heuristic blend, not a trained or calibrated prediction:
# 50% semantic similarity, 30% skill coverage, 20% TF-IDF lexical similarity.
# If the skill dictionary finds no job skills, redistribute the weights across text signals.
if job_skills_canonical:
    overall_match_score = (
        0.50 * semantic_similarity_pct
        + 0.30 * skill_coverage
        + 0.20 * tfidf_similarity_pct
    )
else:
    overall_match_score = 0.70 * semantic_similarity_pct + 0.30 * tfidf_similarity_pct

overall_match_score = float(np.clip(overall_match_score, 0, 100))

score_summary = pd.DataFrame([{
    "Semantic similarity (%)": round(semantic_similarity_pct, 2),
    "TF-IDF similarity (%)": round(tfidf_similarity_pct, 2),
    "Skill coverage (%)": round(skill_coverage, 2),
    "Overall heuristic score (%)": round(overall_match_score, 2),
    "Matched skills": len(matched_skills),
    "Potential skill gaps": len(missing_skills),
}])
score_summary

## 16. Visualize the matching signals

In [ ]:
labels = ["Semantic similarity", "TF-IDF similarity", "Skill coverage", "Overall score"]
values = [semantic_similarity_pct, tfidf_similarity_pct, skill_coverage, overall_match_score]

plt.figure(figsize=(9, 4.5))
bars = plt.bar(labels, values)
plt.ylim(0, 105)
plt.ylabel("Score (%)")
plt.title("Resume-to-Job Matching Signals")
plt.xticks(rotation=12, ha="right")
for bar, value in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f"{value:.1f}%", ha="center", va="bottom")
plt.tight_layout()
plt.show()

## 17. Inspect matched and missing skills

In [ ]:
skills_table = pd.DataFrame({
    "Skill": sorted(set(job_skills_canonical) | set(resume_skills_canonical), key=str.lower)
})
skills_table["In resume"] = skills_table["Skill"].isin(resume_skills_canonical)
skills_table["Required by job description"] = skills_table["Skill"].isin(job_skills_canonical)
skills_table["Status"] = np.select(
    [
        skills_table["In resume"] & skills_table["Required by job description"],
        ~skills_table["In resume"] & skills_table["Required by job description"],
        skills_table["In resume"] & ~skills_table["Required by job description"],
    ],
    ["Matched", "Potential gap", "Additional resume skill"],
    default="Not identified"
)
display(skills_table)

## 18. Export results

In [ ]:
result = {
    "resume_file": resume_path.name,
    "email": candidate_info.get("email"),
    "phone": candidate_info.get("phone"),
    "links": candidate_info.get("links", []),
    "resume_skills": resume_skills_canonical,
    "job_skills": job_skills_canonical,
    "matched_skills": matched_skills,
    "missing_skills": missing_skills,
    "extra_skills": extra_skills,
    "skill_coverage_percent": round(skill_coverage, 2),
    "tfidf_similarity_percent": round(tfidf_similarity_pct, 2),
    "semantic_similarity_raw_cosine": round(semantic_similarity, 4),
    "semantic_similarity_mapped_percent": round(semantic_similarity_pct, 2),
    "overall_heuristic_score_percent": round(overall_match_score, 2),
    "section_semantic_cosine_scores": section_scores,
    "semantic_model": SEMANTIC_MODEL_NAME,
    "score_note": "Heuristic relevance score, not a calibrated hiring probability."
}

output_path = Path("resume_match_results.json")
with output_path.open("w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

skills_table.to_csv("skill_comparison.csv", index=False)
print("Saved:", output_path.resolve())
print("Saved:", Path("skill_comparison.csv").resolve())

## 19. Optional: compare several job descriptions

Use this when you want to see which of several roles is semantically closest to the resume. Paste each job description as a separate item. The result is a shortlist aid, not a hiring recommendation.

In [ ]:
# Example template. Replace these with real job descriptions.
job_options = {
    "ML Engineer": job_description,
    "Data Analyst": "Analyze datasets with SQL, Python, pandas, visualization, and statistics. Communicate insights to stakeholders.",
    "Backend Python Developer": "Build backend services using Python, FastAPI, REST APIs, SQL databases, testing, and Git.",
}

multi_job_rows = []
for role, description in job_options.items():
    role_embedding = document_embedding(description)
    role_semantic = float(np.dot(resume_embedding, role_embedding))
    role_semantic_pct = float(np.clip(role_semantic, 0, 1) * 100)
    role_skills = canonical_skill_set(find_skills(description))
    role_coverage = (
        len(set(resume_skills_canonical) & set(role_skills)) / len(set(role_skills)) * 100
        if role_skills else np.nan
    )
    multi_job_rows.append({
        "Role": role,
        "Semantic similarity (%)": round(role_semantic_pct, 2),
        "Skill coverage (%)": round(role_coverage, 2) if not np.isnan(role_coverage) else None,
    })

job_ranking = pd.DataFrame(multi_job_rows).sort_values(
    "Semantic similarity (%)", ascending=False
).reset_index(drop=True)
display(job_ranking)

## 20. Limitations, evaluation, and responsible use

- **Semantic similarity is not a probability.** Sentence embeddings can capture related language, but may miss role-specific requirements or produce false matches.
- **Skill extraction is vocabulary-limited.** Extend `SKILLS` and `SKILL_ALIASES`; validate extraction against manually labeled examples.
- **Section parsing is heuristic.** Unusual headings, columns, and design-heavy resumes can confuse it.
- **PDF extraction may fail on scans.** OCR is not included in this notebook.
- **Evaluation needed:** Create a small set of resume/job pairs labeled by a human as relevant/not relevant. Compare rankings with Precision@K, Recall@K, and nDCG; evaluate extracted skills with precision, recall, and F1.
- **Privacy:** Use resumes only with appropriate permission, avoid logging unnecessary personal information, and delete test files when finished.
- **Fairness:** Do not use sensitive personal attributes or protected characteristics to rank candidates. Treat the tool as decision support and keep human review central.